## Derry Girls Subrredit scraper and Sentiment analysis

-------


###### This project entailed getting the qualitative feedback of viewers based on the ending of the series Derry Girls. For this reason there is a section of this code which filters for posts mentoining the ending specfically so that the sentiment analysis is only conducted on said specfic posts. 

###### Praw allows you to scrape within a certain period based on when you use the code, for this reason results may vary if executed today
###### This code was originally executed on Feburary 3rd

In [ ]:
import requests
import pandas as pd
from datetime import datetime
import praw

# Set up Reddit API (PRAW)
# Enter your own information after creating an app via https://www.reddit.com/prefs/apps
reddit = praw.Reddit(
    client_id='######',
    client_secret='#######',
    user_agent='######'
)

# Function to pull recent posts using PRAW
def fetch_praw_posts(subreddit_name, limit=1000):
    praw_posts = []
    subreddit = reddit.subreddit(subreddit_name)

    for submission in subreddit.new(limit=limit):
        praw_posts.append({
            'post_id': submission.id,
            'title': submission.title,
            'author': str(submission.author),
            'created_utc': datetime.fromtimestamp(submission.created_utc),
            'url': submission.url,
            'score': submission.score,
            'num_comments': submission.num_comments,
            'selftext': submission.selftext,
            'permalink': "https://www.reddit.com" + submission.permalink
        })

    return praw_posts

# --- Configuration ---
subreddit_name = "DerryGirls"
start_date = "2024-01-01"
end_date = "2024-12-31"

# Run both scrapers
praw_data = fetch_praw_posts(subreddit_name, limit=1000)
print(f"Fetched {len(praw_data)} posts via PRAW")

df = pd.DataFrame(praw_data)

# Save to CSV
df.to_csv("DerryGirls.csv", index=False)
print(f"Saved {len(df)} total posts to 'DerryGirls.csv'")


Fetched 999 posts via PRAW
Saved 999 total posts to 'DerryGirls.csv'


In [3]:
# Remove stop words from titles and selftext using NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Define a function to remove stop words
def remove_stop_words(text):
    if not isinstance(text, str):
        return ""
    try:
        stop_words = set(stopwords.words('english'))
        words = word_tokenize(text)
        filtered_words = [word for word in words if word.lower() not in stop_words]
        return ' '.join(filtered_words)
    except Exception as e:
        print(f"Error processing text: {e}")
        return text

# Apply stop word removal to titles and selftext in PRAW data
for post in praw_data:
    if isinstance(post, dict):
        post['title'] = remove_stop_words(post.get('title', ''))
        post['selftext'] = remove_stop_words(post.get('selftext', ''))


# Create a DataFrame from the updated combined_data
df = pd.DataFrame(praw_data)

# Save the updated DataFrame to CSV
df.to_csv("DerryGirls Cleaned.csv", index=False)
print(f"Saved {len(df)} total posts to 'DerryGirls Cleaned.csv'")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saved 999 total posts to 'DerryGirls Cleaned.csv'


In [ ]:
# Filter selftext for posts containing specific words

# Define keywords to filter for
keywords = ['finale', 'end', 'ending']

# Load the cleaned CSV
df_cleaned = pd.read_csv("DerryGirls Cleaned.csv")

# Function to check if any keyword is in the text
def contains_keywords(text, keywords):
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

# Filter posts where selftext contains any of the keywords
df_filtered = df_cleaned[df_cleaned['selftext'].apply(lambda x: contains_keywords(x, keywords))]

# Save filtered data to CSV
df_filtered.to_csv("DerryGirls_Filtered_Finale_End.csv", index=False)
print(f"Found {len(df_filtered)} posts containing keywords: {', '.join(keywords)}")
print(f"Saved to 'DerryGirls_Filtered_Finale_End.csv'")

# Display a sample
print("\nSample of filtered posts:")
print(df_filtered[['title', 'selftext']].head())


Found 126 posts containing keywords: finale, end, ending
Saved to 'DerryGirls_Filtered_Finale_End.csv'

Sample of filtered posts:
                                    title  \
7                 discovered Derry Girls…   
18                           Struck prime   
23    Derry actually queer friendly 90s ?   
30  Nicola Coughlan 'Derry Girls ' Ending   
32  Ulster Project alums lurking thread ?   

                                             selftext  
7   .. past Sunday already finished Season 1 . lov...  
18      Sister Emily died tragically tender age 104 .  
23  come Asian country idea culture Europe except ...  
30  [ https : //www.bustle.com/entertainment/nicol...  
32  n't know - Ulster Project ( ? ) sort student e...  


In [ ]:
from datetime import datetime
import time

reddit = praw.Reddit(
    client_id='######',
    client_secret='######',
    user_agent='#######'
)


stop_words = set(stopwords.words('english'))

def remove_stop_words(text):
    if not isinstance(text, str):
        return ""
    words = word_tokenize(text)
    filtered = [w for w in words if w.lower() not in stop_words and w.isalpha()]
    return ' '.join(filtered)

Saved 2913 comments from 2024.


In [ ]:


def scrape_comments(subreddit_name, limit=1000):
    comments_data = []
    subreddit = reddit.subreddit(subreddit_name)
    
    # Get recent submissions
    for submission in subreddit.new(limit=limit):
        # Replace MoreComments objects with actual comments
        submission.comments.replace_more(limit=0)
        
        # Process each comment
        for comment in submission.comments.list():
            try:
                comment_data = {
                    'post_id': submission.id,
                    'post_title': submission.title,
                    'comment_id': comment.id,
                    'comment_body': remove_stop_words(comment.body),
                    'comment_author': str(comment.author),
                    'score': comment.score,
                    'created_utc': datetime.fromtimestamp(comment.created_utc)
                }
                comments_data.append(comment_data)
            except Exception as e:
                print(f"Error processing comment: {e}")
                continue
            
        print(f"Processed {len(comments_data)} comments so far...")
        
    # Create DataFrame and save to CSV
    df_comments = pd.DataFrame(comments_data)
    df_comments.to_csv(f"{subreddit_name}_Comments_Latest.csv", index=False)
    print(f"Saved {len(df_comments)} comments to CSV")
    
    return df_comments

# Call the function
df_comments = scrape_comments(subreddit_name)
subreddit_name = "DerryGirls"

Processed 14 comments so far...
Processed 54 comments so far...
Processed 74 comments so far...
Processed 77 comments so far...
Processed 77 comments so far...
Processed 104 comments so far...
Processed 137 comments so far...
Processed 140 comments so far...
Processed 191 comments so far...
Processed 234 comments so far...
Processed 251 comments so far...
Processed 283 comments so far...
Processed 291 comments so far...
Processed 338 comments so far...
Processed 352 comments so far...
Processed 362 comments so far...
Processed 362 comments so far...
Processed 367 comments so far...
Processed 441 comments so far...
Processed 470 comments so far...
Processed 503 comments so far...
Processed 517 comments so far...
Processed 519 comments so far...
Processed 543 comments so far...
Processed 649 comments so far...
Processed 681 comments so far...
Processed 706 comments so far...
Processed 707 comments so far...
Processed 717 comments so far...
Processed 731 comments so far...
Processed 734 c

In [24]:
# Remove stop words from titles and selftext using NLTK
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Define a function to remove stop words
def remove_stop_words(text):
    if not isinstance(text, str):
        return ""
    try:
        stop_words = set(stopwords.words('english'))
        words = word_tokenize(text)
        filtered_words = [word for word in words if word.lower() not in stop_words]
        return ' '.join(filtered_words)
    except Exception as e:
        print(f"Error processing text: {e}")
        return text


for comment in comments_data:
    if isinstance(comment, dict):
        comment['title'] = remove_stop_words(comment.get('title', ''))
        comment['selftext'] = remove_stop_words(comment.get('selftext', ''))

# Save the updated DataFrame to CSV
df_comments.to_csv("DerryGirls Cleaned comments.csv", index=False)
print(f"Saved {len(df_comments)} total comments to 'DerryGirls Cleaned comments.csv'")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\finar\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saved 27540 total comments to 'DerryGirls Cleaned comments.csv'


In [27]:
#Filter selftext for posts containing specific words

# Define keywords to filter for
keywords = ['finale', 'end', 'ending']

# Load the cleaned CSV
df_cleanedcomments = pd.read_csv("DerryGirls Cleaned comments.csv")

# Function to check if any keyword is in the text
def contains_keywords(text, keywords):
    if not isinstance(text, str):
        return False
    text_lower = text.lower()
    return any(keyword.lower() in text_lower for keyword in keywords)

# Filter posts where selftext contains any of the keywords
df_filtered = df_cleanedcomments[df_cleanedcomments['comment_body'].apply(lambda x: contains_keywords(x, keywords))]

# Save filtered data to CSV
df_filtered.to_csv("DerryGirls_Filtered_Finale_End_Comments.csv", index=False)
print(f"Found {len(df_filtered)} posts containing keywords: {', '.join(keywords)}")
print(f"Saved to 'DerryGirls_Filtered_Finale_End_Comments.csv'")

# Display a sample
print("\nSample of filtered posts:")
print(df_filtered[['post_title', 'comment_body']].head())


Found 1846 posts containing keywords: finale, end, ending
Saved to 'DerryGirls_Filtered_Finale_End_Comments.csv'

Sample of filtered posts:
            post_title                                       comment_body
82  The Usual Suspects  Highly recommend joke scene near end obvs miss...
83  The Usual Suspects  saw Derry Girls great movie show ruined twist end
88  The Usual Suspects  Never heard movie show movie really good got f...
89  The Usual Suspects                girls age watched theatre Recommend
90  The Usual Suspects  watched one favorites feel like common movies ...
